# Notebook 16 — Routing, Retries, Admission Control, and Load Shedding

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/16_routing_retries_admission_control_and_load_shedding.ipynb)

Serving systems fail in ordinary ways: queues grow, retries multiply load, and low-value work crowds out urgent requests. This notebook studies those behaviors with notebook-local simulations so you can see which control policies stabilize the runtime and which ones make overload worse.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What you will build**
- Understand **Why this topic belongs after speculation and scheduling**
- Understand **Step 1: Add notebook helpers and an artifact directory**
- Understand **Step 2: Define backend tiers and policy knobs**
- Understand **Step 3: Generate a bursty traffic stream**


## What you will build

- a bursty multi-tenant traffic stream with critical, standard, and batch requests
- a routing simulator with multiple backend tiers and fallback paths
- blind-retry, bounded-retry, admission-control, and load-shedding policies
- per-tenant fairness summaries and shed-reason analysis
- an operational artifact describing the least-bad overload policy for this workload

## Why this topic belongs after speculation and scheduling

Faster decode paths and smarter batching help until traffic exceeds what the runtime can honestly absorb. Once that happens, the important questions become policy questions: which requests get admitted, which ones retry, which ones fail fast, and which ones fall back to a cheaper or smaller backend.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add notebook helpers and an artifact directory

The simulations below are intentionally simple but operationally realistic enough to expose retry amplification, overloaded queues, and the value of graceful degradation.

In [ ]:
random.seed(16)

ARTIFACT_DIR = ARTIFACT_ROOT / "16_routing_retries_admission_control_and_load_shedding"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def percentile(values, pct):
    values = [float(v) for v in values]
    if not values:
        return 0.0
    return float(np.percentile(values, pct))

def clipped(value, low, high):
    return max(low, min(high, value))

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define backend tiers and policy knobs

The backends are deliberately different: a slower high-quality pool, a balanced middle tier, and a cheap fallback tier. Each policy decides how aggressively to retry, when to reject, and whether to shed low-priority work.

In [ ]:
backends = {
    "primary_70b": {"prefill_tps": 8400, "decode_tps": 44, "failure_bias": 0.05},
    "balanced_14b": {"prefill_tps": 10800, "decode_tps": 82, "failure_bias": 0.035},
    "cheap_7b": {"prefill_tps": 12600, "decode_tps": 118, "failure_bias": 0.025},
}

traffic_profiles = [
    {"workload": "assistant_chat", "base_prompt_tokens": 1100, "base_output_tokens": 180, "priority": "interactive", "backend_order": ["primary_70b", "balanced_14b", "cheap_7b"]},
    {"workload": "tool_json", "base_prompt_tokens": 760, "base_output_tokens": 110, "priority": "critical", "backend_order": ["balanced_14b", "primary_70b", "cheap_7b"]},
    {"workload": "batch_summary", "base_prompt_tokens": 5400, "base_output_tokens": 150, "priority": "batch", "backend_order": ["balanced_14b", "cheap_7b", "primary_70b"]},
    {"workload": "safety_review", "base_prompt_tokens": 1500, "base_output_tokens": 125, "priority": "critical", "backend_order": ["primary_70b", "balanced_14b", "cheap_7b"]},
    {"workload": "ops_triage", "base_prompt_tokens": 680, "base_output_tokens": 95, "priority": "interactive", "backend_order": ["balanced_14b", "cheap_7b", "primary_70b"]},
]

policy_configs = [
    {"name": "blind_retries", "max_attempts": 3, "retry_backoff_ms": 0, "jitter_ms": 0, "blind_retry": True, "admission_control": False, "load_shedding": False, "retry_other_backend": False, "queue_budget_ms": 999999, "failed_attempt_cost_ms": 28},
    {"name": "bounded_retries", "max_attempts": 2, "retry_backoff_ms": 35, "jitter_ms": 25, "blind_retry": False, "admission_control": False, "load_shedding": False, "retry_other_backend": True, "queue_budget_ms": 999999, "failed_attempt_cost_ms": 24},
    {"name": "admission_control", "max_attempts": 1, "retry_backoff_ms": 20, "jitter_ms": 10, "blind_retry": False, "admission_control": True, "load_shedding": False, "retry_other_backend": True, "queue_budget_ms": 650, "failed_attempt_cost_ms": 20},
    {"name": "load_shedding", "max_attempts": 1, "retry_backoff_ms": 18, "jitter_ms": 8, "blind_retry": False, "admission_control": True, "load_shedding": True, "retry_other_backend": True, "queue_budget_ms": 520, "failed_attempt_cost_ms": 18},
]

pd.DataFrame(policy_configs)[["name", "max_attempts", "queue_budget_ms", "load_shedding"]]

## Step 3: Generate a bursty traffic stream

The arrival process includes traffic spikes because overload behavior is rarely visible under smooth synthetic load.

In [ ]:
tenants = [
    {"tenant": "castalia_mentor", "mix": [0.42, 0.16, 0.12, 0.14, 0.16]},
    {"tenant": "ops_console", "mix": [0.18, 0.12, 0.08, 0.34, 0.28]},
    {"tenant": "research_lab", "mix": [0.20, 0.18, 0.24, 0.12, 0.26]},
    {"tenant": "nightly_batch", "mix": [0.08, 0.06, 0.58, 0.08, 0.20]},
]

def choose_profile(weights, rng):
    roll = rng.random()
    cursor = 0.0
    for index, weight in enumerate(weights):
        cursor += weight
        if roll <= cursor:
            return traffic_profiles[index]
    return traffic_profiles[-1]

records = []
arrival_clock = 0.0
for index in range(260):
    tenant_profile = tenants[index % len(tenants)]
    rng = random.Random(f"traffic-{index}")
    burst_gap = rng.uniform(1.5, 6.0) if index % 21 else rng.uniform(18.0, 32.0)
    arrival_clock += burst_gap
    profile = choose_profile(tenant_profile["mix"], rng)
    prompt_tokens = max(90, int(rng.gauss(profile["base_prompt_tokens"], profile["base_prompt_tokens"] * 0.16)))
    output_tokens = max(24, int(rng.gauss(profile["base_output_tokens"], profile["base_output_tokens"] * 0.18)))
    timeout_ms = 1600 if profile["priority"] in {"interactive", "critical"} else 2600
    records.append({
        "request_id": f"req_{index:04d}",
        "tenant": tenant_profile["tenant"],
        "arrival_ms": round(arrival_clock, 2),
        "workload": profile["workload"],
        "priority": profile["priority"],
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "timeout_ms": timeout_ms,
        "backend_order": list(profile["backend_order"]),
    })

traffic_df = pd.DataFrame(records)
traffic_df.head(10)

## Step 4: Inspect the traffic shape before routing

A router should be tuned against the traffic it actually sees: request value, prompt length, and timeout tolerance all matter.

In [ ]:
traffic_mix = (
    traffic_df.groupby(["tenant", "priority"])
    .agg(
        requests=("request_id", "count"),
        avg_prompt_tokens=("prompt_tokens", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
    )
    .round(2)
    .reset_index()
)
traffic_mix.head(12)

## Step 5: Implement the routing simulator

Every request chooses a backend, may wait in queue, may fail, and may retry or fall back depending on policy. Failed attempts still consume some backend time, which is how retries amplify overload.

In [ ]:
def service_ms(record, backend_name):
    backend = backends[backend_name]
    prefill_ms = 1000 * record["prompt_tokens"] / backend["prefill_tps"]
    decode_ms = 1000 * record["output_tokens"] / backend["decode_tps"]
    quality_penalty_ms = 22 if backend_name == "primary_70b" else (14 if backend_name == "balanced_14b" else 8)
    return prefill_ms + decode_ms + quality_penalty_ms

def run_policy(records, config):
    state = {name: {"busy_until": 0.0} for name in backends}
    outputs = []
    for record in sorted((dict(row) for row in records), key=lambda item: item["arrival_ms"]):
        attempt_arrival = record["arrival_ms"]
        attempts = 0
        candidate_index = 0
        final_status = "failed"
        final_backend = "none"
        final_latency_ms = 0.0
        final_queue_ms = 0.0
        fallback_used = False
        shed_reason = ""

        while attempts < config["max_attempts"] and candidate_index < len(record["backend_order"]):
            backend_name = record["backend_order"][0] if config["blind_retry"] else record["backend_order"][candidate_index]
            backend_state = state[backend_name]
            queue_ms = max(0.0, backend_state["busy_until"] - attempt_arrival)
            attempts += 1

            if config["admission_control"] and queue_ms > config["queue_budget_ms"]:
                if config["load_shedding"] and record["priority"] == "batch":
                    final_status = "shed"
                    shed_reason = "shed_batch_queue_budget"
                    final_backend = backend_name
                    final_latency_ms = queue_ms
                    final_queue_ms = queue_ms
                    break
                if config["load_shedding"] and record["priority"] == "interactive" and candidate_index + 1 < len(record["backend_order"]):
                    candidate_index += 1
                    fallback_used = True
                    attempt_arrival += config["retry_backoff_ms"]
                    continue
                if record["priority"] != "critical":
                    final_status = "rejected"
                    shed_reason = "admission_reject"
                    final_backend = backend_name
                    final_latency_ms = queue_ms
                    final_queue_ms = queue_ms
                    break

            service = service_ms(record, backend_name)
            overload_prob = clipped(backends[backend_name]["failure_bias"] + queue_ms / 3600 + record["prompt_tokens"] / 140000, 0.01, 0.88)
            rng = random.Random(f'{config["name"]}-{record["request_id"]}-{attempts}-{backend_name}')
            if rng.random() < overload_prob:
                backend_state["busy_until"] = max(backend_state["busy_until"], attempt_arrival) + config["failed_attempt_cost_ms"]
                final_backend = backend_name
                final_queue_ms = queue_ms
                final_latency_ms = max(0.0, attempt_arrival - record["arrival_ms"]) + config["failed_attempt_cost_ms"]
                if attempts >= config["max_attempts"]:
                    final_status = "failed"
                    shed_reason = "backend_overload"
                    break
                if config["retry_other_backend"] and candidate_index + 1 < len(record["backend_order"]):
                    candidate_index += 1
                    fallback_used = True
                backoff = config["retry_backoff_ms"] + rng.uniform(0, config["jitter_ms"])
                attempt_arrival += backoff
                continue

            start_ms = max(attempt_arrival, backend_state["busy_until"])
            finish_ms = start_ms + service
            backend_state["busy_until"] = finish_ms
            final_status = "success"
            final_backend = backend_name
            final_queue_ms = queue_ms
            final_latency_ms = finish_ms - record["arrival_ms"]
            break

        timed_out = final_latency_ms > record["timeout_ms"] and final_status == "success"
        if timed_out:
            final_status = "timeout"
            shed_reason = "client_timeout"

        outputs.append({
            **record,
            "policy": config["name"],
            "attempts": attempts,
            "success": final_status == "success",
            "final_status": final_status,
            "backend": final_backend,
            "fallback_used": fallback_used,
            "latency_ms": round(final_latency_ms, 2),
            "queue_ms": round(final_queue_ms, 2),
            "shed_reason": shed_reason,
            "served_output_tokens": record["output_tokens"] if final_status == "success" else 0,
        })
    return pd.DataFrame(outputs)

traffic_records = traffic_df.to_dict("records")

## Step 6: Run the blind-retry baseline

Blind retries are seductive because they are easy to add. Under overload they often make the system worse by adding extra attempt traffic to the already saturated pool.

In [ ]:
blind_df = run_policy(traffic_records, policy_configs[0])
blind_df[["request_id", "priority", "backend", "attempts", "final_status", "latency_ms"]].head(12)

## Step 7: Add bounded retries with jitter and cross-backend fallback

Bounded retries assume a failure might be local to one backend or a brief spike. They still add load, but much less than unbounded or immediate retries.

In [ ]:
bounded_df = run_policy(traffic_records, policy_configs[1])
bounded_df[["request_id", "priority", "backend", "attempts", "fallback_used", "final_status", "latency_ms"]].head(12)

## Step 8: Add admission control

Admission control does something uncomfortable but healthy: it says no when the queue budget is already blown. That prevents the runtime from pretending it can meet an SLO that it has already missed.

In [ ]:
admission_df = run_policy(traffic_records, policy_configs[2])
admission_df[["request_id", "priority", "backend", "attempts", "final_status", "shed_reason", "latency_ms"]].head(12)

## Step 9: Add load shedding and fallback

Load shedding goes one step further. It sacrifices low-value work so the runtime can protect higher-value requests. In practice that often means batch jobs are shed first and interactive jobs are rerouted to a smaller backend.

In [ ]:
shedding_df = run_policy(traffic_records, policy_configs[3])
shedding_df[["request_id", "priority", "backend", "attempts", "fallback_used", "final_status", "shed_reason", "latency_ms"]].head(12)

## Step 10: Compare the policies directly

The summary below is the operational heart of the notebook: success rate, retry amplification, fallback rate, and tail latency.

In [ ]:
def summarize_policy(frame):
    success_frame = frame[frame["success"]].copy()
    busy_window_s = max((frame["arrival_ms"].max() - frame["arrival_ms"].min()) / 1000, 0.001)
    return {
        "policy": frame["policy"].iloc[0],
        "success_rate": round(frame["success"].mean(), 4),
        "p95_latency_ms": round(percentile(success_frame["latency_ms"], 95), 2),
        "retry_amplification": round(frame["attempts"].sum() / len(frame), 3),
        "fallback_rate": round(frame["fallback_used"].mean(), 4),
        "reject_or_shed_rate": round((frame["final_status"].isin(["rejected", "shed"])).mean(), 4),
        "critical_success_rate": round(frame[frame["priority"] == "critical"]["success"].mean(), 4),
        "throughput_tps": round(frame["served_output_tokens"].sum() / busy_window_s, 2),
    }

policy_summary = pd.DataFrame([
    summarize_policy(blind_df),
    summarize_policy(bounded_df),
    summarize_policy(admission_df),
    summarize_policy(shedding_df),
]).sort_values("critical_success_rate", ascending=False)
policy_summary

## Step 11: Visualize the amplification problem

A policy that looks generous can still be destructive if it creates more attempts than the system can digest. Retry amplification is often the clearest explanation for overload spirals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
policy_summary.plot(x="policy", y=["p95_latency_ms", "critical_success_rate"], kind="bar", rot=0, ax=axes[0], color=["#f58518", "#54a24b"])
policy_summary.plot(x="policy", y="retry_amplification", kind="bar", rot=0, ax=axes[1], color="#e45756", legend=False)
axes[0].set_title("Latency and critical-path success")
axes[0].set_ylabel("value")
axes[1].set_title("Retry amplification per request")
axes[1].set_ylabel("attempts / request")
plt.tight_layout()

## Step 12: Measure fairness by tenant

Overload policy is also a fairness policy. If one tenant or one priority class absorbs all the pain, that should be visible.

In [ ]:
combined_df = pd.concat([blind_df, bounded_df, admission_df, shedding_df], ignore_index=True)
tenant_summary = (
    combined_df.groupby(["policy", "tenant"])
    .agg(
        requests=("request_id", "count"),
        success_rate=("success", "mean"),
        p95_latency_ms=("latency_ms", lambda values: percentile(values, 95)),
        mean_attempts=("attempts", "mean"),
    )
    .round(3)
    .reset_index()
)
tenant_summary.head(16)

## Step 13: Inspect shed reasons and fallback distribution

A good overload policy should be explainable. We want to know whether work was rejected by queue budget, timed out, or deliberately shed because it was low priority.

In [ ]:
shedding_reasons = (
    shedding_df.groupby(["final_status", "shed_reason"])
    .size()
    .rename("requests")
    .reset_index()
    .sort_values("requests", ascending=False)
)

fallback_distribution = (
    shedding_df[shedding_df["fallback_used"]]
    .groupby("backend")
    .size()
    .rename("requests")
    .reset_index()
    .sort_values("requests", ascending=False)
)

print(shedding_reasons.to_markdown(index=False))
print(fallback_distribution.to_markdown(index=False))

## Step 14: Export an operational policy artifact

Later notebooks can use this artifact to decide which overload policy should gate rollout or route selection.

In [ ]:
recommended_policy = policy_summary.sort_values(["critical_success_rate", "p95_latency_ms", "retry_amplification"], ascending=[False, True, True]).iloc[0]["policy"]
artifact = {
    "notebook": "16_routing_retries_admission_control_and_load_shedding",
    "policy_summary": policy_summary.to_dict("records"),
    "tenant_summary": tenant_summary.to_dict("records"),
    "shedding_reasons": shedding_reasons.to_dict("records"),
    "fallback_distribution": fallback_distribution.to_dict("records"),
    "recommended_policy": recommended_policy,
}

artifact_path = ARTIFACT_DIR / "overload_policy_summary.json"
write_json(artifact_path, artifact)
print("Recommended policy:", recommended_policy)
print("Wrote:", artifact_path.resolve())

## Recap

Retry logic is not neutral. Blind retries often turn transient overload into a self-inflicted incident. Admission control and load shedding feel harsher, but they keep the system honest by protecting the requests that still have a chance to succeed within budget.

In [ ]:
summary_rows = len(policy_summary)
assert summary_rows == 4
assert policy_summary["retry_amplification"].max() >= policy_summary["retry_amplification"].min()
assert tenant_summary["tenant"].nunique() == len(tenants)
print("Notebook 16 sanity checks passed.")

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** benchmark with a different model size and compare throughput. Document what changes and why.

**Exercise 2 — Build:** add a monitoring metric to the serving pipeline. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** simulate a failure scenario and verify the system recovers.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the systems/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [15 Speculative Decoding And Assisted Generation](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/15_speculative_decoding_and_assisted_generation.ipynb) | ➡️ **Next:** [17 Distributed Inference Concepts](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/17_distributed_inference_concepts.ipynb)